# Algorithm laboratory
Edit `algorithms.py` or `my_algorithm.py`, rerun the import cell, and inspect every saved step.
This notebook uses CPU by default. Install the optional extras with `pip install -e '.[laboratory]'`.
Inspection and timing are separate passes. All numbers are also available as JSON/NumPy files.


In [ ]:
from pathlib import Path
import sys, importlib, uuid
import numpy as np
import torch
from threadpoolctl import threadpool_limits
from IPython.display import display, Image

for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "svdbench").is_dir():
        ROOT = candidate
        sys.path.insert(0, str(ROOT))
        break
else:
    raise RuntimeError("Open the notebook from this repository")
from svdbench.laboratory import MatrixCase, Trace, NullTrace, Run, accuracy, benchmark, basis_quality, residual_arrays, check_accuracy
from algorithm_laboratory import algorithms, my_algorithm
importlib.reload(algorithms)
importlib.reload(my_algorithm)
torch.set_num_threads(4)
torch.backends.cuda.matmul.allow_tf32 = False


## Choose the specimen and the experiment
Change shape, dtype, generating spectrum or algorithm parameters. You can instead use
`MatrixCase.load(ROOT / "experiments/broad-20261012/dense1024/matrix.npy")`.
`MatrixCase.from_spectrum([5, 5, 1, 0.1, 0], m=12, n=8)` prepares a custom spectrum.


In [ ]:
DEVICE = "cpu"                    # "cuda:0" on a GPU machine
RANK = 4
ITERATIONS = 3
SEED = 7
case = MatrixCase.create(m=48, n=32, family="gaussian", dtype="float64", seed=23)
a = case.tensor(DEVICE)
case.spec, case.sha256, case.reference[0][1]


## Inspect each iteration
The numerical implementation is in `algorithms.randomized`; edit its loops or supply your own
`orthogonalizer` / `core_solver`. Use `my_algorithm.solve` for the blank working starter.
You can insert `breakpoint()` into either Python file. Stage timers surround computation;
checkpoints copy/validate outside those timers.


In [ ]:
RUN_DIR = ROOT / "laboratory_runs" / ("notebook-" + uuid.uuid4().hex[:8])
with threadpool_limits(4), torch.inference_mode():
    with Trace(RUN_DIR, case, DEVICE, parameters={"rank": RANK, "iterations": ITERATIONS, "seed": SEED},
               source_files=[algorithms.__file__, my_algorithm.__file__]) as trace:
        factors = algorithms.randomized(a, trace, rank=RANK, iterations=ITERATIONS, seed=SEED)
        trace.checkpoint("elementwise_errors", tensors=residual_arrays(case, factors))
    metrics = accuracy(case, factors)
    del factors
run = Run(RUN_DIR)
display(run.dataframe("checkpoint"))


## Touch the actual intermediate arrays
Select any checkpoint by event ID, name or iteration. No arrays are reconstructed from summary
statistics. When a snapshot was omitted by the storage budget, `run.tensor()` tells you why.
Compare subspace projectors instead of individual signs/phases of singular vectors.


In [ ]:
steps = run.select(kind="checkpoint", name="iterate")
q0 = run.tensor(steps[0]["event_id"], "Q")
q1 = run.tensor(steps[1]["event_id"], "Q")
print("First Q entries:", q0[:6, :4])
print("Subspace projector change:", np.linalg.norm(q1 @ q1.conj().T - q0 @ q0.conj().T))
display(steps[1]["accuracy"]["worst"])
display(basis_quality(case, q1))


In [ ]:
# Individual singular-value errors and entrywise residuals remain available.
last = steps[-1]
s = run.tensor(last["event_id"], "s")
relative_errors = run.tensor(last["event_id"], "relative_error")
print("Singular values:", s)
print("Relative errors:", relative_errors)
residual_event = run.select(name="elementwise_errors")[0]
residual = run.tensor(residual_event["event_id"], "reconstruction_residual")
print("Residual block:", residual[:6, :6])
display(check_accuracy(metrics, residual_over_optimal_tail=1.05,
                       resolved_spectrum_max_relative_error=.01))
display(Image(filename=str(run.plot())))


## Measure the algorithm with inspection disabled
This runs the same implementation with `NullTrace`, omitting diagnostic projected SVDs,
CPU copies and checkpoints. Repetitions measure timing variation for the same sketch seed.
Use `study.py` for separate sketch-seed trials and parameter sweeps.


In [ ]:
with threadpool_limits(4):
    timing = benchmark(lambda: algorithms.randomized(a, NullTrace(), rank=RANK,
                        iterations=ITERATIONS, seed=SEED), device=DEVICE, repeat=7, warmup=2)
display(timing)


## Experiments to make next
- Replace only `orthogonalize` with a different QR/reorthogonalization method and compare Q Gram errors.
- Compare q=0,1,2,4 using the same matrix and sketch seed; look at each leading-value error.
- Run Jacobi with `max_sweeps=1, require_convergence=False`, then 2, 3, ... .
- Add `record_rotations=True` to Jacobi to inspect W/V after each independent group of rotations.
- Create exactly repeated singular values; observe when a leading subspace is not uniquely defined.
- Replace a dense product with a COO product; keep the sketch fixed.
- Use `python -m algorithm_laboratory.parts` for an isolated operation benchmark.
- Use `--power-seconds 5` on the NVIDIA experiment CLI for a separate sustained power pass.

See `docs/ALGORITHM_LABORATORY.md` for measurement contracts and running on a GPU machine.
